In [1]:
%pip install langchain langchain-community pypdf langchain-text-splitters langchain-qdrant langchain-huggingface qdrant-client


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:


from langchain_community.document_loaders import TextLoader

file = "data/nlp_article.txt"

document = TextLoader(file_path=file)
loader = document.load()

print(f"Document Loaded Successfully")
print(f"Total Documents {len(loader)}")

C:\Users\Muhammad Hamza\AppData\Local\Temp\ipykernel_2860\3538373058.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
C:\Users\Muhammad Hamza\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Document Loaded Successfully
Total Documents 1


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(loader)

print(f"Data is successfully splitted into chunks")
print(f"Total Chunks Into Data {len(chunks)} " )


Data is successfully splitted into chunks
Total Chunks Into Data 21 


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient


QDRANT_URL=""
QDRANT_API_KEY=""

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

if client.collection_exists(collection_name="model_testing"):
    client.delete_collection(collection_name="model_testing")
    print("Delete Old Collection Successfully")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectors = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="model_testing"
)

print(f"collection is created successfully ")
print(f"Please See your qdrant dashboard")



Delete Old Collection Successfully


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2495.14it/s]


collection is created successfully 
Please See your qdrant dashboard


In [6]:
query="What do you think about NLP?"

retriever = vectors.as_retriever()
results = retriever.invoke(query)

for i,data in enumerate(results):
    print(f" Index No {i+1}")
    print(f" Page Content \n {data.page_content}")
    print("="*70)

 Index No 1
 Page Content 
 NLP combines computational linguistics â€” rule-based modeling of human language â€” with statistical, machine learning, and deep learning models. Together, these technologies enable computers to process human language in the form of text or voice data and to understand its full meaning, complete with the speaker's or writer's intent and sentiment.
 Index No 2
 Page Content 
 Natural Language Processing: A Comprehensive Guide

Chapter 1: What is Natural Language Processing?

Natural Language Processing, commonly known as NLP, is a branch of artificial intelligence that focuses on the interaction between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, understand, and make sense of human language in a manner that is both valuable and meaningful.
 Index No 3
 Page Content 
 Consider ethical implications. NLP systems can perpetuate biases present in training data. Be aware of potential biases in your models and 

In [11]:
%pip install -U langchain-google-genai


  Using cached pyasn1-0.6.3-py3-none-any.whl.metadata (8.4 kB)
   ---------------------------------------- 0.0/958.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/958.0 kB ? eta -:--:--
   ---------------------------------------- 958.0/958.0 kB 3.6 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ------------------------ --------------- 2.4/3.8 MB 12.0 MB/s eta 0:00:01
   --------------------------------- ------ 3.1/3.8 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 6.7 MB/s  0:00:00
Using cached pyasn1-0.6.3-py3-none-any.whl (83 kB)

   ---------------------------------------- 0/7 [filetype]
   ---------------------------------------- 0/7 [filetype]
   ----- ---------------------------------- 1/7 [pyasn1]
   ----- ---------------------------------- 1/7 [pyasn1]
   ----- ---------------------------------- 1/7 [pyasn1]
   ----- ---------------------------------- 1/7 [pyasn1]
   ----------- ----------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os 
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

os.environ["GOOGLE_API_KEY"] = ""

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.2)

template="""You are an expert AI Assistant representing Corex Tech. 
Use the following pieces of retrieved context to answer the user's question professionally. 
If you do not know the answer based on the context, politely say that Sorry I don't have enough information to give you answer. 
Keep the answer concise and clear.

Context:
{context}

Question: {question}

Helpful Answer:"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 5. Core LCEL Chain Structure (The Magic Line )
# Note: 'retriever' aap ka existing Qdrant MMR retriever object hai
lcel_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("🔗 LCEL Custom RAG Chain successfully compiled!")


🔗 LCEL Custom RAG Chain successfully compiled!


In [8]:
# Sawaal poochen
user_query = "What do you think about NLP?"

# Chain invoke karein (LCEL direct string output deta hai)
final_answer = lcel_rag_chain.invoke(user_query)

print("🤖 Gemini LCEL Output:\n")
print(final_answer)


🤖 Gemini LCEL Output:

At Corex Tech, we view Natural Language Processing (NLP) as a powerful branch of artificial intelligence that enables computers to process, understand, and make sense of human language—both text and voice—including its full meaning, intent, and sentiment. It achieves this by combining computational linguistics with statistical, machine learning, and deep learning models. 

To successfully implement NLP, we emphasize that data quality is paramount, requiring thorough cleaning and preprocessing. Additionally, it is critical to address ethical implications by actively mitigating biases in training data and ensuring strict compliance with privacy regulations like GDPR and CCPA.


In [9]:
# Sawaal poochen
user_query = "What is hope to skill"

# Chain invoke karein (LCEL direct string output deta hai)
final_answer = lcel_rag_chain.invoke(user_query)

print("🤖 Gemini LCEL Output:\n")
print(final_answer)

🤖 Gemini LCEL Output:

Sorry, I don't have enough information to give you an answer.


In [10]:
# Sawaal poochen
user_query = "ya NLP kya hai?"

# Chain invoke karein (LCEL direct string output deta hai)
final_answer = lcel_rag_chain.invoke(user_query)

print("🤖 Gemini LCEL Output:\n")
print(final_answer)

🤖 Gemini LCEL Output:

Hello! On behalf of Corex Tech, I'd be happy to explain. 

**NLP (Natural Language Processing)** artificial intelligence (AI) ki ek branch hai jo computers aur humans ke beech natural language ke zariye interaction par focus karti hai. 

Context ke mutabiq, NLP computational linguistics ko statistical, machine learning, aur deep learning models ke sath combine karta hai. Iska main objective computers ko insani bhasha (text ya voice data) ko padhne, samajhne, aur uske sahi matlab (intent aur sentiment) ko decipher karne ke kabil banana hai.
